# Resurvey: re-run only `construction_trim_split_*` variants

`construction_trim_split_5_5` / `_8_2` recently switched from
`type4 (bestsofar_feb2023 construction)` + `type6` to `type5 + type6` (both via
the trained edit_model). The CSVs in `artifacts/results/` were computed under
the old implementation. This notebook **surgically replaces only the rows
belonging to those two variants** in every affected file, without re-running
the rest of the experiments.

For each affected dataset we:

1. Load the existing `*_routes.pt` to get the unified list of `RunResult`s
   and the saved coords/street_adj.
2. Extract `init_routes` from the `Initial network` run's `seed_routes`
   (saved alongside).
3. Re-run only `construction_trim_split_5_5` / `_8_2` (both accept modes
   where applicable) via the same `run_method` the original notebook used.
4. Replace those entries in the `.pt`, re-save the `.pt`.
5. Rebuild the comparison table via `build_comparison_table` and the
   seed-sweep summary via `_summarize_seed_sweep` -- saving back into the
   same CSV names.

Sections are independent -- if a setup ingredient is missing locally
(`datasets/raw_graphs_1000.pkl` for the NX dataset is the usual blocker),
that section logs a warning and is skipped; the rest still run.

The CSV write goes through `save_table`, which **overwrites** the existing
file at `artifacts/results/<name>.csv`. A `.bak` copy of every file we are
about to overwrite is made first (the cell below). Old construction_trim_split
results are intentionally lost when the new ones land.


In [ ]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

import os as _os, sys as _sys
_NB_DIR = _os.path.abspath(".")
if _NB_DIR not in _sys.path:
    _sys.path.insert(0, _NB_DIR)
import eval_lib
from eval_lib import *
from eval_lib import plots as _plots
from eval_lib import _run_baseline  # private, skipped by import *
from eval_lib.sweep import _summarize_seed_sweep  # ditto, needed for the summary CSVs
from eval_lib.tables import run_result_row

from connectpt.routes_generator import CityGraphData, build_nx_heuristic_routes
from connectpt.routes_generator.citygraph_dataset import load_macsa_scenarios

# Default tensors for make_test_dataloader (mirrors eval notebook cell 6).
set_input_tensors(load_benchmark_tensors())

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

RECOMPUTE_KEYS = {"construction_trim_split_5_5", "construction_trim_split_8_2"}
RECOMPUTE_VARIANTS = [v for v in BCO_VARIANTS if v["key"] in RECOMPUTE_KEYS]
RECOMPUTE_LABELS = {v["summary_label"] for v in RECOMPUTE_VARIANTS}
print("Will recompute variants:", [v["key"] for v in RECOMPUTE_VARIANTS])
print("Labels (CSV `method` column):", sorted(RECOMPUTE_LABELS))


In [ ]:
# --- safety: back up every CSV/.pt we will overwrite ---
TARGET_CSVS = [
    "mumford0_lc_comparison",
    "worse_accept_lc_summary",
    "nx_dataset_comparison",
    "nx_heuristic_dataset_comparison",
    "worse_accept_nx_dataset_summary",
    "macsa_comparison",
    "macsa_summary",
    "benchmark_sweep",
]
TARGET_PTS = [
    "mumford0_lc_routes_without_worse",
    "mumford0_lc_routes_worse",
    "nx_dataset_routes_without_worse",
    "nx_dataset_routes_worse",
    "macsa_mandl_8_without_worse",
    "macsa_mandl_8_worse",
    "benchmark_Mandl",
    "benchmark_Mumford0",
    "benchmark_Mumford1",
    "benchmark_Mumford2",
    "benchmark_Mumford3",
]

for stem in TARGET_CSVS:
    src = RESULTS_DIR / f"{stem}.csv"
    if src.exists():
        shutil.copy(src, src.with_suffix(".csv.bak"))
for stem in TARGET_PTS:
    src = RESULTS_DIR / f"{stem}_routes.pt"
    if src.exists():
        shutil.copy(src, src.with_suffix(".pt.bak"))
print("Backups created with .bak extension.")


In [ ]:
# --- shared helpers ---

VARIANT_ORDER = [v["key"] for v in BCO_VARIANTS]
LABEL_BY_KEY = {v["key"]: v["summary_label"] for v in BCO_VARIANTS}


def rerun_recompute(*, init_routes, tensors, accept_modes,
                    n_routes, min_route_len, max_route_len,
                    dataset_label, run_name_scope, seed=0):
    """Re-run only the construction_trim_split variants on (init_routes, tensors)
    for the given accept_modes. Returns list[RunResult]."""
    out = []
    for variant in RECOMPUTE_VARIANTS:
        spec = bco_method(variant)
        for accept_mode in accept_modes:
            print(f"  -> {spec['label']} ({accept_mode})")
            try:
                r = run_method(
                    spec, init_routes=init_routes, tensors=tensors,
                    n_routes=n_routes, min_route_len=min_route_len,
                    max_route_len=max_route_len, accept_mode=accept_mode,
                    seed=seed, dataset=dataset_label,
                    run_name_scope=run_name_scope,
                )
            except Exception as exc:
                print(f"     FAILED: {exc!r}")
                continue
            out.append(r)
    return out


def replace_entries(existing_runs, new_runs):
    """Return list with BCO entries matching RECOMPUTE_LABELS replaced by
    new_runs (matched by (label, accept_mode))."""
    new_by_key = {(r.label, r.accept_mode): r for r in new_runs
                  if r.kind == "bco" and r.label in RECOMPUTE_LABELS}
    out = []
    matched = set()
    for r in existing_runs:
        key = (r.label, r.accept_mode)
        if key in new_by_key:
            out.append(new_by_key[key])
            matched.add(key)
        else:
            out.append(r)
    missed = set(new_by_key) - matched
    if missed:
        print(f"  WARN: new runs not matched in existing .pt: {missed}")
    return out


def interleave_seed_sweep_order(runs_ww, runs_w):
    """Re-build the unified RunResult list in run_seed_sweep order:
    Initial, RL_only, then per-variant {without_worse, worse}."""
    out = []
    init = next((r for r in runs_ww if r.kind == "initial"), None)
    if init is not None:
        out.append(init)
    rl = next((r for r in runs_ww if r.kind == "rl_only"), None)
    if rl is not None:
        out.append(rl)
    for vkey in VARIANT_ORDER:
        label = LABEL_BY_KEY[vkey]
        ww = next((r for r in runs_ww
                   if r.kind == "bco" and r.label == label), None)
        w  = next((r for r in runs_w
                   if r.kind == "bco" and r.label == label), None)
        if ww is not None: out.append(ww)
        if w  is not None: out.append(w)
    return out


def build_summary_df(results, graph_index=0, seed=0):
    rows = []
    for r in results:
        row = run_result_row(r)
        row["graph_index"] = graph_index
        row["seed"] = seed
        rows.append(row)
    return _summarize_seed_sweep(pd.DataFrame(rows))


In [ ]:
# === §9 LC: Mumford0 LC init ===
try:
    runs_ww, coords_lc, street_lc = load_route_results(
        "mumford0_lc_routes_without_worse")
    runs_w, _, _ = load_route_results("mumford0_lc_routes_worse")
    lc_base_routes = next(r.seed_routes for r in runs_ww if r.kind == "initial")
    print(f"§9 LC: lc_base_routes shape = {tuple(lc_base_routes.shape)}")

    new = rerun_recompute(
        init_routes=lc_base_routes, tensors=None,
        accept_modes=("without_worse", "worse"),
        n_routes=N_ROUTES, min_route_len=MIN_ROUTE_LEN,
        max_route_len=MAX_ROUTE_LEN,
        dataset_label="Mumford0 LC init",
        run_name_scope="worse_accept_resurvey_")

    new_ww = [r for r in new if r.accept_mode == "without_worse"]
    new_w  = [r for r in new if r.accept_mode == "worse"]

    updated_ww = replace_entries(runs_ww, new_ww)
    updated_w  = replace_entries(runs_w,  new_w)

    save_route_results(updated_ww, "mumford0_lc_routes_without_worse",
                       coords=coords_lc, street_adj=street_lc)
    save_route_results(updated_w,  "mumford0_lc_routes_worse",
                       coords=coords_lc, street_adj=street_lc)

    unified = interleave_seed_sweep_order(updated_ww, updated_w)
    save_table(build_comparison_table(unified), "mumford0_lc_comparison")
    save_table(build_summary_df(unified), "worse_accept_lc_summary")
    print("§9 LC: updated mumford0_lc_comparison.csv + worse_accept_lc_summary.csv")
except Exception as exc:
    print(f"§9 LC: SKIPPED ({exc!r})")


In [ ]:
# === §9 NX dataset + §8b summary ===
# We need nx_sweep_tensors (node_locs, street_adj, demand). Only coords +
# street_adj sit inside the saved .pt -- the demand tensor lives in the
# pickle dataset. If raw_graphs_1000.pkl is missing locally, skip cleanly.
NX_SWEEP_GRAPH_INDEX = 0  # matches eval notebook cell de3281a6

try:
    raw_graphs_path = DATASETS_DIR / "raw_graphs_1000.pkl"
    if not raw_graphs_path.exists():
        raise FileNotFoundError(raw_graphs_path)

    nx_graph, nx_sweep_initial_routes, _ = load_single_nx_heuristic_graph_and_routes(
        raw_graphs_path, LC_RESULTS_DIR, NX_SWEEP_GRAPH_INDEX)
    nx_sweep_tensors = nx_graph_to_tensor_dataset(nx_graph)
    print(f"§9 NX: graph={NX_SWEEP_GRAPH_INDEX} loaded")

    runs_ww, coords_nx, street_nx = load_route_results(
        "nx_dataset_routes_without_worse")
    runs_w, _, _ = load_route_results("nx_dataset_routes_worse")

    new = rerun_recompute(
        init_routes=nx_sweep_initial_routes, tensors=nx_sweep_tensors,
        accept_modes=("without_worse", "worse"),
        n_routes=N_ROUTES, min_route_len=MIN_ROUTE_LEN,
        max_route_len=MAX_ROUTE_LEN,
        dataset_label=f"NX dataset graph {NX_SWEEP_GRAPH_INDEX}",
        run_name_scope="nx_dataset_resurvey_")

    new_ww = [r for r in new if r.accept_mode == "without_worse"]
    new_w  = [r for r in new if r.accept_mode == "worse"]
    updated_ww = replace_entries(runs_ww, new_ww)
    updated_w  = replace_entries(runs_w,  new_w)

    save_route_results(updated_ww, "nx_dataset_routes_without_worse",
                       coords=coords_nx, street_adj=street_nx)
    save_route_results(updated_w,  "nx_dataset_routes_worse",
                       coords=coords_nx, street_adj=street_nx)

    unified = interleave_seed_sweep_order(updated_ww, updated_w)
    save_table(build_comparison_table(unified), "nx_dataset_comparison")
    save_table(build_comparison_table(unified), "nx_heuristic_dataset_comparison")
    save_table(build_summary_df(unified), "worse_accept_nx_dataset_summary")
    print("§9 NX: updated comparison + nx_heuristic_dataset_comparison + summary")
except Exception as exc:
    print(f"§9 NX: SKIPPED ({exc!r})")


In [ ]:
# === §11 MACSA ===
try:
    macsa_scenarios = load_macsa_scenarios(MACSA_DATA_DIR)
    print(f"§11 MACSA: loaded {len(macsa_scenarios)} scenario(s)")

    for scenario in macsa_scenarios:
        name = scenario["name"]
        stem_ww = f"macsa_{name}_without_worse"
        stem_w  = f"macsa_{name}_worse"
        try:
            runs_ww, coords_m, street_m = load_route_results(stem_ww)
            runs_w, _, _ = load_route_results(stem_w)
        except FileNotFoundError as exc:
            print(f"  scenario {name}: skipped ({exc!r})")
            continue

        n_routes, min_len, max_len = macsa_eval_bounds(scenario)
        new = rerun_recompute(
            init_routes=scenario["routes"], tensors=scenario["tensors"],
            accept_modes=("without_worse", "worse"),
            n_routes=n_routes, min_route_len=min_len, max_route_len=max_len,
            dataset_label=name,
            run_name_scope=f"macsa_resurvey_{name}_")

        new_ww = [r for r in new if r.accept_mode == "without_worse"]
        new_w  = [r for r in new if r.accept_mode == "worse"]
        updated_ww = replace_entries(runs_ww, new_ww)
        updated_w  = replace_entries(runs_w,  new_w)

        save_route_results(updated_ww, stem_ww, coords=coords_m, street_adj=street_m)
        save_route_results(updated_w,  stem_w,  coords=coords_m, street_adj=street_m)

        unified = interleave_seed_sweep_order(updated_ww, updated_w)
        # macsa_comparison.csv is one DataFrame across all scenarios -- here
        # there is a single scenario, so build it directly. If more scenarios
        # were stored, would need to concat across them.
        save_table(build_comparison_table(unified), "macsa_comparison")
        save_table(build_summary_df(unified), "macsa_summary")
        print(f"  scenario {name}: updated")
except Exception as exc:
    print(f"§11 MACSA: SKIPPED ({exc!r})")


In [ ]:
# === §12 benchmark (Mandl + Mumford0..3) ===
# Per-city `benchmark_<city>_routes.pt` contains 1 reference + ~11 cases
# (BCO variants + RL + SA + GA + HH + NSGA). We replace the 2 construction_trim
# entries, save back, and update `benchmark_sweep.csv` rows.
try:
    benchmark_df = pd.read_csv(RESULTS_DIR / "benchmark_sweep.csv")
    new_rows_by_city = {}
    for spec in BENCHMARK_SPECS:
        city = spec["city"]
        try:
            runs_city, coords_c, street_c = load_route_results(
                f"benchmark_{city}")
        except FileNotFoundError as exc:
            print(f"  {city}: SKIPPED ({exc!r})")
            continue
        init_routes = next(r.routes for r in runs_city if r.kind == "initial")

        new = rerun_recompute(
            init_routes=init_routes,
            tensors=load_benchmark_tensors(city),
            accept_modes=("without_worse",),
            n_routes=spec["n_routes"],
            min_route_len=spec["min_route_len"],
            max_route_len=spec["max_route_len"],
            dataset_label=city,
            run_name_scope=f"benchmark_resurvey_{city}_")

        # benchmark .pt entries are saved with label = "BCO: <variant_label>".
        # Build a RunResult list matching that naming so replace_entries fires.
        relabelled = []
        for r in new:
            new_r = RunResult(
                label=f"BCO: {r.label}", kind=r.kind, accept_mode=r.accept_mode,
                dataset=r.dataset, metrics=r.metrics, routes=r.routes,
                seed_routes=r.seed_routes, unserved_demand=r.unserved_demand,
                mutation_stats=r.mutation_stats, action_stats=r.action_stats,
                weights=r.weights, run_name=r.run_name)
            relabelled.append(new_r)

        bench_targets = {f"BCO: {lbl}" for lbl in RECOMPUTE_LABELS}
        new_by_lbl = {r.label: r for r in relabelled}
        updated = []
        for r in runs_city:
            if r.label in bench_targets and r.label in new_by_lbl:
                updated.append(new_by_lbl[r.label])
            else:
                updated.append(r)
        save_route_results(updated, f"benchmark_{city}",
                           coords=coords_c, street_adj=street_c)

        # Build the benchmark CSV rows the same way summarize_benchmark_run does.
        city_new_rows = [
            summarize_benchmark_run(city, r.label, r.metrics) for r in relabelled
        ]
        new_rows_by_city[city] = city_new_rows
        print(f"  {city}: regenerated {len(city_new_rows)} row(s)")

    # Replace rows in benchmark_sweep.csv
    if new_rows_by_city:
        mask_remove = (
            (benchmark_df["benchmark"].isin(new_rows_by_city.keys())) &
            (benchmark_df["method"].isin({f"BCO: {lbl}" for lbl in RECOMPUTE_LABELS}))
        )
        kept = benchmark_df[~mask_remove].copy()
        added = pd.concat([pd.DataFrame(rs) for rs in new_rows_by_city.values()],
                          ignore_index=True)
        # Preserve original column order; align to existing columns.
        added = added.reindex(columns=benchmark_df.columns)
        updated_df = pd.concat([kept, added], ignore_index=True)
        # Restore per-city grouping order in the file (kept rows stay in place;
        # new rows append at the bottom). Sort by benchmark (stable) so each
        # city block is contiguous.
        cat_order = pd.Categorical(updated_df["benchmark"],
                                   categories=[s["city"] for s in BENCHMARK_SPECS],
                                   ordered=True)
        updated_df = updated_df.assign(_o=cat_order).sort_values(
            by="_o", kind="stable").drop(columns="_o").reset_index(drop=True)
        save_table(updated_df, "benchmark_sweep")
        print("§12 benchmark: updated benchmark_sweep.csv")
except Exception as exc:
    print(f"§12 benchmark: SKIPPED ({exc!r})")


In [ ]:
# === verification: show the replaced rows in each CSV ===
for stem in ["mumford0_lc_comparison", "nx_dataset_comparison",
             "macsa_comparison", "benchmark_sweep"]:
    path = RESULTS_DIR / f"{stem}.csv"
    if not path.exists():
        continue
    df = pd.read_csv(path)
    method_col = "method"
    if method_col not in df.columns:
        continue
    bench_labels = {f"BCO: {lbl}" for lbl in RECOMPUTE_LABELS}
    mask = df[method_col].isin(RECOMPUTE_LABELS) | df[method_col].isin(bench_labels)
    sub = df[mask].copy()
    if sub.empty:
        continue
    print(f"\n----- {stem}.csv ({len(sub)} replaced rows) -----")
    cols = [c for c in
            ["benchmark", "dataset", "method", "accept_mode",
             "cost", "ATT", "RTT", "d_un", "$d_{un}$",
             "Cp (ATT)", "Co (RTT)"] if c in sub.columns]
    display(sub[cols].round(4))
